# Plan the trajectory and analyse the flight

Use Day 1's tracking and Day 2's uncertainty to plan, compare controller settings and interpret recorded flights.

| Practical | Minutes | Keep |
| --- | --- | --- |
| R1: course and roles | 15 | Agreed rules, limits and stop operator. |
| R2: shared survey | 40 + 20 retry buffer | Checked gate geometry. |
| R3: plan and gain choice | 35 | Margin budget and one controller-gain prediction. |
| R4: evidence and dry run | 30 | One result from each earlier day and a checked plan. |
| R5: low-speed check | 30 shared | Configuration-specific go/no-go. |
| R6: race | 100 shared | Configuration, telemetry and observed crossings. |
| R7: analysis | 35 | One plot, explanation and next change. |

**Getting started.** Start `jupyter lab` from the project root and select **Crazyflie lectures (.venv)**; the printed Python path should be inside `.venv`. Run cells with **Shift+Enter**. Paths beginning with `root /` are relative to the project root.

**Run All** uses a simulated course and a labelled delayed-reference recording. Prefer checked surveys and actual telemetry; after five minutes of acquisition trouble, keep the files and use shared data or the offline example.

Choose a fresh `run_id` for another session (`FileExistsError` protects existing results). After editing an input, rerun from that cell through export. Use [01_survey.ipynb](01_survey.ipynb) for gate acquisition and [03_flight.ipynb](03_flight.ipynb) for the flight tool's dry run and flight.

In [ ]:
from pathlib import Path
from copy import deepcopy
import sys

# Find the same project and environment used on Days 1 and 2.
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "code/day3/plan.py").is_file())
sys.path.insert(0, str(root / "code"))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from trajectory_tools.course import configure_crossings, construct_course, read_json
from trajectory_tools.conservative import slow_defaults
from day3.notebook_tools import (
    plan_course, plan_race, checked_plan, save_plan, clearance_budget, save_example_replay,
)
from trajectory_tools.preview import plot_trajectory
from race_common.flight_analysis import analyze_recording

run_id = "trial-01"  # Change before repeating setup.
output = root / "code/day3/data" / run_id
output.mkdir(parents=True, exist_ok=False)
print("Python:", sys.executable)
print("Results:", output)


## 1. The course and coordinate frame (R1)

Use the agreed right-handed Lighthouse frame: `[x, y, z]` in metres, Z upward. A gate's unit normal points perpendicular to its opening. Predict the sign for the first crossing.

Select `template_path` below. The default is **simulated back-and-forth motion**; measured courses need measured endpoints and agreed limits.

| Setting | Meaning |
| --- | --- |
| `start`, `finish` | Airborne positions at rest; separate from the resting place for takeoff/landing. |
| `bounds_min`, `bounds_max` | XYZ limits including netting, floor and supports. |
| `target_speed_m_s`, `maximum_speed_m_s` | Requested speed and shared upper limit. |
| `maximum_acceleration_m_s2`, `maximum_tilt_deg` | Acceleration and nominal bank limits. |
| `drone_radius_m`, `clearance_margin_m` | Full cage/deck envelope plus survey, estimation and tracking allowance. |
| `pre_post_distance_m` | Entry/exit distance along the gate normal. |
| `setpoint_rate_hz` | Reference samples per second. |

Edit `template["limits"]` before its display. Retain checked yaw, tracking/telemetry thresholds and takeoff/landing durations. Keep shared physical limits fixed.

**Check:** agree rules and roles, including a separate emergency-stop operator. Day 1's position loop produced velocity commands; the race runner sends world XYZ position targets to onboard control. Gains belong to their command interface.

In [ ]:
baseline_plan, candidate_plan = None, None
template, course = None, None
template_path = root / "examples/course.shuttle.synthetic.json"
template = read_json(template_path)
# For a measured course, edit start/finish and the agreed limits here.
# Example: template["limits"]["pre_post_distance_m"] = 0.35
display({key: template[key] for key in ("course_name", "start", "finish", "enabled_for_flight")})
display(template["limits"])


## 2. Measured gates and crossing order (R2)

Use [01_survey.ipynb](01_survey.ipynb) for a timed trace and a fitted circle or rectangle. It verifies `stabilizer.stop=1` and zero motor outputs. Wait for **STOP VERIFIED** before handling; follow its preparation and recording-start messages.

Share one trace and an independent check per gate. Check centre, normal and aperture against manual measurements or a second pass. Rectangles use an inscribed circle for planning. Apply a documented fixed-attitude pointer offset to a copy of the survey; retain the original.

Paste `survey_paths` below. `hoop-1:+` follows the normal; `hoop-1:-` opposes it. Both directions with `laps = 2` give four crossings. Empty lists use the simulated template.

**Check:** a second group member checks geometry, direction and clearance. Keep shared checked geometry for retries. Place the drone down and **power-cycle** before flight preflight.


In [ ]:
baseline_plan, candidate_plan = None, None
course = None  # Failed input edits invalidate downstream plans too.
course_provenance = None
survey_paths = []  # Example: [root / "survey/hoop-1.json"]
crossing_order = []  # Example: ["hoop-1:+", "hoop-1:-"]
laps = 2
conservative_mode = True  # Keep for the first course; False selects custom planning.
# Defaults: cruise <=0.20 m/s, peak <=0.30 m/s, acceleration <=0.50 m/s2, tilt <=10 deg.
# Entry/exit stops and automatic bypasses retain the shared flight volume.
# Frame depth defaults to 0.10 m; check it against the actual gates/supports.
target_speed_m_s = None  # Default: at most 0.20 m/s requested cruise.
gate_speeds = {}  # m/s; e.g. {"forward": 0.25, "return": 0.15}.
crossing_offsets = {}  # World XYZ metres in the gate plane; {"forward": [0, 0.05, 0]}.
# Keys above are gate IDs (all laps) or exact crossing names (one passage).
turning_waypoints = {}  # Completed crossings -> XYZ stops; e.g. {1: [[1.6, 0.4, 0.7]]}.
# Fly-through in custom mode: {1: [{"position": [1.7, 0.4, 0.7], "velocity": [0, -0.1, 0]}]}.
# Velocity constrains the trajectory derivative; the runner streams position targets.
# Slot 0 is before crossing 1; slot N follows crossing N, across the full race.

surveys = [read_json(path) for path in survey_paths]
for survey in surveys:
    display({key: survey[key] for key in ("name", "centre", "normal", "radius_m",
                                          "shape", "width_m", "height_m") if key in survey})
print("Geometry:", "selected survey files; independent checks still needed" if surveys else "TEMPLATE ONLY; no measured survey selected")

constructed_course = construct_course(template, surveys, crossing_order, laps)
if conservative_mode:
    constructed_course = slow_defaults(constructed_course)
course = configure_crossings(constructed_course, gate_speeds=gate_speeds,
                             crossing_offsets=crossing_offsets, turning_waypoints=turning_waypoints)
course_provenance = {
    "source_template": str(template_path),
    "source_surveys": [str(path) for path in survey_paths],
    "crossing_order": list(crossing_order),
    "laps": laps,
}
print("Ordered crossings:", [gate["name"] for gate in course["hoops"]])
print("Enabled for flight:", course["enabled_for_flight"])


**Entry, crossing and exit.** With centre C, in-plane offset o, normal n and distance d, cross at C+o; entry/exit are C+o-d*n and C+o+d*n. Turning waypoints default to zero velocity.

Radial space is radius minus drone envelope, margin and offset magnitude. Check the allowance against earlier evidence below.

In [ ]:
gate = course["hoops"][0]
centre = np.asarray(gate["centre"])
normal = np.asarray(gate["normal"])
distance = course["limits"]["pre_post_distance_m"]
offset = np.asarray(gate.get("crossing_offset_m", [0, 0, 0]))
crossing_point = centre + offset
entry = crossing_point - distance * normal
exit_point = crossing_point + distance * normal
for label, point in (("entry", entry), ("centre", centre), ("crossing", crossing_point), ("exit", exit_point)):
    print(f"{label:6s} [m]: {point}")
radial_space = (gate["radius_m"] - course["limits"]["drone_radius_m"]
                - course["limits"]["clearance_margin_m"] - np.linalg.norm(offset))
print(f"Radial space after drone and margin: {radial_space:.3f} m")


## 3. A continuous trajectory (R3)

Use one result from Day 1's `2Dsim_04_velocity_loop` or `drone_05_tuning`, and one from Day 2 E3-E7, to choose the allowances below. Starting values are illustrative. Day 2's height filter supplies vertical uncertainty; horizontal allowances need separate evidence or stated assumptions.

**Margin.** C is the Day 2 **state-estimation error covariance**; C_position is its position block in m^2. For gate-plane basis U, use U.T @ C_position @ U. The largest eigenvalue gives the widest-axis variance, as in E7:

```text
required margin = survey allowance + tracking allowance + k * plane standard deviation
```

The `our_gate_margin` function below projects the covariance and supplies each gate's planning allowance. U has two orthonormal columns spanning the opening plane. The helper validates inputs and rejects results below the projected uncertainty plus the chosen allowances.

Tracking allowance covers additional control lag/bias; account for overlap with estimation effects. A shadow filter's C needs justification before use for onboard estimates. The sum is a conservative allowance, not an exact collision probability. The planner retains the shared minimum margin and rejects an envelope that cannot fit.

**Motion.** Quintic segments join entries, crossings, exits and turns with continuous position, velocity and acceleration (C2). Conservative mode stops at entries/exits and routes around gate envelopes; crossings remain moving. Start/finish and zero-velocity waypoints are at rest. Time stretching by s divides speed by s and acceleration by s^2. The summary shows resulting speeds; jerk can jump at joins.
Turning velocities use world XYZ m/s; omitted values give stops. The table compares requested and time-scaled reference velocities.

In the preview, grey openings and red normals show crossings; dashed orange envelopes show finite-frame bypass regions. `frame_margin_m` defaults to 0.10 m per side, with drone radius and clearance added. Check supports and raised floors separately. A bypass is not a scored crossing.

**Predict:** which gate consumes most margin, and where does acceleration peak? Printed bounds cover the continuous curve; sampled peaks may be lower. Duration is planned-reference time, excluding takeoff/landing. Check the flight runner's own dry run too.
Use Day 1's response plots to predict one horizontal P-gain change in [03_flight.ipynb](03_flight.ipynb). Keep course/speed fixed for both gain trials. Kalman tuning and the speed preview below are optional.


In [ ]:
baseline_plan, candidate_plan = None, None  # Failed edits invalidate both plans.

def our_gate_margin(U, C_position, survey_m, tracking_m, k):
    C_plane = U.T @ C_position @ U
    plane_std_m = np.sqrt(max(0.0, np.linalg.eigvalsh(C_plane).max()))
    required_margin_m = survey_m + tracking_m + k * plane_std_m
    return plane_std_m, required_margin_m

# Replace these illustrative allowances with justified values from your earlier work.
position_std_m = np.array([0.01, 0.01, 0.015])
C_position = np.diag(position_std_m**2)
budget = clearance_budget(
    course, C_position, survey_allowance_m=0.01,
    tracking_allowance_m=0.03, sigma_multiplier=2.5,
    margin_calculation=our_gate_margin,
)
for gate_budget in budget["gates"]:
    print(gate_budget)

# Keep the margin calculation and motion choices visible; the API validates them.
clearance_m = max(
    course["limits"]["clearance_margin_m"], budget["required_margin_m"])
approach_distance_m = course["limits"]["pre_post_distance_m"]
planning_provenance = dict(course_provenance, clearance_budget=budget)
baseline_plan = plan_race(course, speed=target_speed_m_s, clearance=clearance_m,
                          approach_distance=approach_distance_m, gate_speeds=gate_speeds,
                          crossing_offsets=crossing_offsets, turning_waypoints=turning_waypoints,
                          conservative=conservative_mode, provenance=planning_provenance)
baseline_course, baseline_trajectory = checked_plan(baseline_plan)
plan_summary = baseline_plan.summary()
display(plan_summary)
for waypoint in plan_summary["turning_details"]:
    requested_velocity = np.asarray(waypoint["requested_velocity_m_s"])
    resulting_velocity = requested_velocity / baseline_trajectory.time_scale
    print("Turning velocity [m/s], requested / resulting:", requested_velocity, resulting_velocity)
figure = baseline_plan.plot()
figure.savefig(output / "baseline.png", dpi=140)
display(figure)
plt.close(figure)
print("Chosen margin [m]:", baseline_course["limits"]["clearance_margin_m"])
print("Planned duration [s]:", baseline_trajectory.duration_s)
print("Conservative speed bound [m/s]:", baseline_trajectory.speed_bound_m_s)
print("Conservative acceleration bound [m/s2]:", baseline_trajectory.acceleration_bound_m_s2)

**Optional speed preview.** Edit `candidate_target_speed` within this plan's maximum, keeping geometry and margin fixed. Predict the duration and identify the limiting bound; different requests may produce the same motion.

Approach-distance changes and jerk continuity are optional extensions after this comparison.

In [ ]:
candidate_plan = None  # A failed rerun must not leave an older exportable plan.
candidate, candidate_trajectory = None, None
if baseline_plan is None:
    raise ValueError("Correct the course inputs and rerun baseline planning before comparing a candidate.")
candidate, baseline_trajectory = checked_plan(baseline_plan)  # Independent copy of the validated course.
candidate_target_speed = 0.25  # m/s; stay within this plan's conservative maximum.
candidate["limits"]["target_speed_m_s"] = candidate_target_speed
candidate_plan = plan_course(candidate, provenance=baseline_plan.provenance)
candidate, candidate_trajectory = checked_plan(candidate_plan)

figure = plot_trajectory(candidate, candidate_trajectory)
figure.savefig(output / "candidate.png", dpi=140)
display(figure)
plt.close(figure)
print("Baseline / candidate reference duration [s]:",
      baseline_trajectory.duration_s, candidate_trajectory.duration_s)


**Compare.** `np.linalg.norm(..., axis=1)` gives XYZ speed/acceleration magnitudes. Inspect straights and turns. What changed in timing, and what headroom remains for feedback? For approximately level flight, bank grows as atan(horizontal acceleration / g).

**Check:** explain a difference and select a plan using duration, margin and headroom.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for label, plan, style in (("baseline", baseline_plan, "-"),
                           ("candidate", candidate_plan, "--")):
    checked_course, trajectory = checked_plan(plan)
    times, positions, velocities, accelerations = trajectory.sample(
        checked_course["limits"]["setpoint_rate_hz"])
    speed = np.linalg.norm(velocities, axis=1)
    acceleration = np.linalg.norm(accelerations, axis=1)
    ax[0].plot(times, speed, linestyle=style, label=label)
    ax[1].plot(times, acceleration, linestyle=style, label=label)
    print(f"{label}: {trajectory.duration_s:.2f} s, "
          f"sampled peak speed {speed.max():.3f} m/s, "
          f"sampled peak acceleration {acceleration.max():.3f} m/s2")
ax[0].axhline(baseline_course["limits"]["maximum_speed_m_s"], color="k", ls="--")
ax[1].axhline(baseline_course["limits"]["maximum_acceleration_m_s2"], color="k", ls="--")
ax[0].set_ylabel("speed [m/s]")
ax[1].set_ylabel("acceleration [m/s2]")
ax[1].set_xlabel("reference time [s]")
for axis in ax:
    axis.grid()
    axis.legend()
fig.tight_layout()
fig.savefig(output / "comparison.png", dpi=140)
display(fig)
plt.close(fig)


## 4. Save the selected plan and check the evidence (R4)

Use the 30-minute slot:

1. **10 minutes:** revisit one Day 1 tracking plot and one Day 2 uncertainty/outlier/dropout plot. Explain the margin or speed choice.
2. **10 minutes:** inspect crossing order, supports, bounds and motion limits; select one plan for both gain trials.
3. **10 minutes:** compare the flight tool's dry-run report with this plan. Record any gap and retain the tested baseline if needed.

`plan_race` returns a validated plan: preview it with `.plot()` and inspect `.summary()`. Open [03_flight.ipynb](03_flight.ipynb) to export both race submissions and dry-run them with the common runner. Compare its report with this preview; shared data remain available offline.

`save_plan` writes disabled `course.json`, `trajectory.csv` (time [s], XYZ position [m], velocity [m/s], acceleration [m/s2]) and `summary.json` (bounds, crossings, input provenance and margin budget). Keep input JSONs and the course/summary pair. Re-export replaces generated files; failed/edited plans need replanning.

**Check:** the dry run establishes configured geometry and limits. The low-speed flight checks physical tracking and visibility.

In [ ]:
selected_plan = baseline_plan  # Conservative default; select candidate_plan after comparison.
summary = save_plan(selected_plan, output)
print("Saved disabled plan:", output / "course.json")
for crossing in summary["crossings"]:
    print(f"{crossing['name']}: {crossing['time_s']:.2f} s")


## 5. The flight (R5-R6)

Use [03_flight.ipynb](03_flight.ipynb) with the run card's runner and checked course. Identify the drone there with **Connect**, then power-cycle and repeat preflight without rerunning Connect. Agree readiness within the group; assign pilot, separate emergency-stop operator, telemetry monitor and recorder.

Cross-check supports/floors, drone envelope, visibility, battery, mass/controller, timing/limits, tracking and landing. Repeat preflight after handling or reboot.

Allow one short low-speed check per ready team, with handovers/recovery. Use the checked baseline or shared data if the candidate/equipment is not ready.

Freeze estimator settings at **12:30**, controller settings at **13:00**, and all algorithm changes at **13:30**. Race **14:00-15:40**, then protect the 35-minute analysis slot. Use the baseline qualification and changed-gain test as the matched comparison. Freeze the set of declared profiles, then select among those checked settings for the remaining race slot.

**Check:** confirm the pilot/stop operator, and save each attempt's exact configuration, telemetry and observed crossings, including aborts. Everyone can call a stop. Onboard estimation/control drives the motors; Day 2's laptop filter is diagnostic in the baseline.

## 6. Recorded motion and tracking difference (R7)

Select telemetry and its **exact flown course** from the attempt folder printed by the flight notebook, or use shared checked data. With both paths empty, the cell creates a labelled example: selected-plan positions delayed by 0.12 s. It prescribes positions for plotting practice rather than predicting drone performance.

CSV `event=position` rows carry `x_m,y_m,z_m`; `event=command` rows carry `command_x_m,command_y_m,command_z_m` and `phase`. The plotter uses `host_monotonic_s`, selects `race`, and matches estimates to the last command, up to 0.25 s old.

The estimated-minus-commanded difference is e(t)=p_onboard(t)-p_command(t). Day 2's predicted residual compares a measurement with its filter prediction. The cell calculates the XYZ differences, their Euclidean magnitudes and the 3D RMS, sqrt(mean(||e||^2)). Unmatched targets are NaN and excluded from those statistics. The helper handles file validation, timestamp alignment and plots. Estimation and timestamp alignment also contribute to e.

Spend **5 minutes selecting data, 20 minutes inspecting the path/errors, and 10 minutes discussing one change**. Compare the baseline and changed-gain records at the same course/speed. Did lag or overshoot change as predicted? Compare turns, straights and clearance; note battery condition. Use observed crossings and timing to judge a race result; identify a single attempt as such.

**Check:** name the data source, explain one feature and keep one plot with your next change.

In [ ]:
telemetry_csv = None  # Paste the flight notebook's printed Recording path.
flown_course_json = None  # Paste the printed Exact flown course archive path.
recording_source = None

if (telemetry_csv is None) != (flown_course_json is None):
    raise ValueError("Select both the telemetry and the exact flown course, or neither.")

if telemetry_csv is None:
    telemetry_path, course_path = save_example_replay(selected_plan, output / "example-replay")
    recording_source = "SYNTHETIC delayed-reference example (0.12 s lag)"
    prefix = output / "synthetic"
else:
    telemetry_path, course_path = Path(telemetry_csv), Path(flown_course_json)
    recording_source = "Selected recording; retain its source and matching course"
    prefix = output / "flown"

print("Data source:", recording_source)
comparison, figures, _ = analyze_recording(telemetry_path, course_path, phase="race")
errors_m = comparison.positions - comparison.targets
matched = np.all(np.isfinite(errors_m), axis=1)
distances_m = np.linalg.norm(errors_m[matched], axis=1)
rms_error_m = np.sqrt(np.mean(distances_m**2))
maximum_error_m = distances_m.max()
print(f"Matched {matched.sum()} / {len(matched)} position samples")
print(f"3D RMS error: {rms_error_m:.3f} m; maximum: {maximum_error_m:.3f} m")
print("Host timing includes transport delay; onboard position is not ground truth.")
for figure, suffix in zip(figures, ("trajectory", "errors")):
    figure.savefig(f"{prefix}-{suffix}.png", dpi=150)
    display(figure)
    plt.close(figure)


## Discuss and keep

Keep the notebook, surveys, selected plan and one useful plot. Add the margin/speed choice, data source, outcome and next change to the team's existing notes.

Declare optional profiles before freezing; test in allocated flight slots. See the [stretch exercises](../../docs/race_practical.md#stretch-exercises).

## If a cell fails

| Symptom | Next step |
| --- | --- |
| Import/root error | Check kernel and project-root launch. |
| Existing output | Choose a fresh `run_id`. |
| Invalid budget/plan | Check units, allowances, geometry and limits; rerun from the edited input through export. |
| Capture or flight check fails | Keep the failed data; use checked shared data, the labelled example or a checked baseline. |

## Reference and optional material

The notebook contains the core steps. Look up these details when needed:
- [Flight checks and deadlines](../../docs/race_practical.md).
- [Survey procedure](../../docs/hoop_survey_challenge.md).
- [Course fields and flight-tool handover](../../docs/course_interface.md).
- [Recording/plotting formats](../../docs/post_flight_analysis.md).

**Optional background and extensions:**
- [Planner mathematics, limitations and jerk-continuity work](../../docs/trajectory_construction.md).
- [Control and estimation equations](../../docs/quadrotor_equations_of_motion.md).